In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [4]:


# Load dataset
df = pd.read_csv("houses_improved.csv")

# -----------------------------
# Encode categorical columns
# -----------------------------
label_encoders = {}

for col in df.select_dtypes(include="object").columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

# -----------------------------
# Features and Target
# -----------------------------
feature_cols = list(df.columns[:-1])      # All columns except the last
target_col = df.columns[-1]               # Last column is target

X_multivariate = df[feature_cols]
y = df[target_col]

print("Selected Features:")
print(feature_cols)

print("\nFeature Matrix Shape:")
print(X_multivariate.shape)

# -----------------------------
# Split Dataset
# -----------------------------
X_train_multi, X_test_multi, y_train, y_test = train_test_split(
    X_multivariate,
    y,
    test_size=0.2,
    random_state=42
)

# -----------------------------
# Feature Scaling
# -----------------------------
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_multi)
X_test_scaled = scaler.transform(X_test_multi)

# -----------------------------
# Train Model
# -----------------------------
multi_model = LinearRegression()

multi_model.fit(X_train_scaled, y_train)

# -----------------------------
# Display Coefficients
# -----------------------------
coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": multi_model.coef_,
    "Abs_Coefficient": np.abs(multi_model.coef_)
}).sort_values(by="Abs_Coefficient", ascending=False)

print("\nMultivariate Linear Regression Model")
print(f"Intercept: {multi_model.intercept_:.2f}")

print("\nFeature Coefficients:")
print(coef_df.to_string(index=False))

# -----------------------------
# Predictions
# -----------------------------
y_pred_train = multi_model.predict(X_train_scaled)
y_pred_test = multi_model.predict(X_test_scaled)

# -----------------------------
# Evaluation
# -----------------------------
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))

train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)

print("\nMultivariate Model Performance")
print(f"Training RMSE: {train_rmse:.2f}")
print(f"Test RMSE: {test_rmse:.2f}")
print(f"Training R² Score: {train_r2:.4f}")
print(f"Test R² Score: {test_r2:.4f}")

C:\Users\user\AppData\Local\Temp\ipykernel_27900\68844494.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


Selected Features:
['Number_of_Rooms', 'Site_Area_sqm', 'Built_Area_sqm', 'Property_Years', 'Construction_Materials', 'Housing_Typology', 'Land_Value_Grading', 'Proximity_to_CBD_km', 'Proximity_to_Bus_Station_km', 'Type_of_Nearest_Road', 'Proximity_to_Schools_km']

Feature Matrix Shape:
(1000, 11)

Multivariate Linear Regression Model
Intercept: 2102110.65

Feature Coefficients:
                    Feature    Coefficient  Abs_Coefficient
             Built_Area_sqm  864605.157107    864605.157107
             Property_Years -465878.678569    465878.678569
        Proximity_to_CBD_km -462553.962258    462553.962258
     Construction_Materials -437813.193443    437813.193443
       Type_of_Nearest_Road -192132.052343    192132.052343
Proximity_to_Bus_Station_km -143131.259525    143131.259525
    Proximity_to_Schools_km -129704.011029    129704.011029
         Land_Value_Grading -126386.528275    126386.528275
           Housing_Typology   58500.257869     58500.257869
              Site

In [5]:
import joblib
import json
import os

os.makedirs("models", exist_ok=True)

# Save model
joblib.dump(multi_model, "models/house_price_model.pkl")

# Save scaler
joblib.dump(scaler, "models/scaler.pkl")

# Save label encoders
joblib.dump(label_encoders, "models/label_encoders.pkl")

# Save feature names
with open("models/features.json", "w") as f:
    json.dump(feature_cols, f)

print("Everything saved successfully!")

Everything saved successfully!
